#RAG PIPELINE





In [12]:
# Build Profiles -- Cell 1

import requests
import re
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/scotus_project'

# Name initialization

CURRENT_JUSTICES = [
    'JGRoberts', 'CThomas', 'SAAlito', 'SSotomayor',
    'EKagan', 'NMGorsuch', 'BMKavanaugh', 'ACBarrett', 'KBJackson'
]

JUSTICE_NAMES = {
    'JGRoberts':  'Chief Justice John Roberts',
    'CThomas':    'Justice Clarence Thomas',
    'SAAlito':    'Justice Samuel Alito',
    'SSotomayor': 'Justice Sonia Sotomayor',
    'EKagan':     'Justice Elena Kagan',
    'NMGorsuch':  'Justice Neil Gorsuch',
    'BMKavanaugh':'Justice Brett Kavanaugh',
    'ACBarrett':  'Justice Amy Coney Barrett',
    'KBJackson':  'Justice Ketanji Brown Jackson'
}

# Manual Profiles (Generated by Claude, reviewed/fact-checked by human (me))

justice_profiles = {
    'JGRoberts': """
Chief Justice John Roberts (appointed by George W. Bush, 2005).
Ideology: Center-right, institutionalist. Martin-Quinn score: 0.71.
Religion: Catholic.
Judicial philosophy: Roberts is a pragmatic conservative who prioritizes
institutional legitimacy of the Supreme Court. He frequently acts as a
swing vote, joining liberals on high-profile cases to avoid 5-4 decisions
that appear partisan. He authored the majority in NFIB v. Sebelius upholding
the ACA, and joined liberals in June Medical Services v. Russo on abortion
access. He is deeply concerned about the Court's public perception and often
seeks narrow rulings that avoid broad constitutional pronouncements.
Key opinions: NFIB v. Sebelius (2012), Shelby County v. Holder (2013),
Department of Homeland Security v. Regents (2020).
Voting patterns: Votes conservative on criminal procedure, business regulation,
and administrative law. More moderate on voting rights, immigration, and
cases threatening Court legitimacy. Rarely dissents alone.
Notable trait: Known as the institutionalist — will sacrifice ideological
consistency to protect the Court as an institution.
""",
    'CThomas': """
Justice Clarence Thomas (appointed by George H.W. Bush, 1991).
Ideology: Far-right originalist. Martin-Quinn score: 3.24.
Religion: Catholic.
Judicial philosophy: Thomas is the Court's most conservative justice and its
most committed originalist. He believes the Constitution should be interpreted
solely based on its original public meaning at ratification. He frequently
writes separate concurrences to argue for overturning precedents he considers
wrongly decided, including substantive due process and administrative deference.
He was the only dissenter in Grutter v. Bollinger and has called for overturning
Lawrence v. Texas and Obergefell.
Key opinions: McDonald v. City of Chicago (2010), Dobbs concurrence (2022),
Bruen (2022).
Voting patterns: Most likely justice to vote alone. Consistently votes against
criminal defendants, affirmative action, abortion rights, and LGBT rights.
Notable trait: Will vote to overturn any precedent he believes was wrongly decided.
""",
    'SAAlito': """
Justice Samuel Alito (appointed by George W. Bush, 2006).
Ideology: Conservative. Martin-Quinn score: 2.54.
Religion: Catholic.
Judicial philosophy: Alito is a results-oriented conservative combining
originalism with traditionalism. He authored Dobbs v. Jackson overturning
Roe v. Wade. He is strongly pro-religion and skeptical of administrative power.
Key opinions: Dobbs v. Jackson (2022), Hobby Lobby (2014), Janus v. AFSCME (2018).
Voting patterns: Reliably conservative on abortion, religion, guns, and
administrative law. Frequently writes sharp dissents when in the minority.
Notable trait: More ideologically driven than Roberts, less concerned with
institutional perception.
""",
    'SSotomayor': """
Justice Sonia Sotomayor (appointed by Barack Obama, 2009).
Ideology: Liberal. Martin-Quinn score: -2.89.
Religion: Catholic.
Judicial philosophy: Sotomayor is the Court's most outspoken liberal justice.
She emphasizes lived experience and empathy in judicial decision-making.
She is a strong advocate for criminal defendants, immigrants, and voting rights.
Key opinions: Utah v. Strieff dissent (2016), Trump v. Hawaii dissent (2018).
Voting patterns: Votes for criminal defendants, immigrants, affirmative action,
abortion rights, and LGBT rights. Almost always votes with Kagan and Jackson.
Notable trait: Most likely to write emotional, accessible dissents aimed at
the public. Will vote to uphold progressive precedents and expand civil rights.
""",
    'EKagan': """
Justice Elena Kagan (appointed by Barack Obama, 2010).
Ideology: Liberal. Martin-Quinn score: -2.17.
Religion: Jewish.
Judicial philosophy: Kagan is a pragmatic liberal and the Court's most skilled
legal craftsperson. As former Harvard Law dean and Solicitor General, she writes
opinions known for clarity. She is a textualist who believes context and purpose
are relevant. She has occasionally persuaded Roberts and Kavanaugh to join narrower rulings.
Key opinions: Yates v. United States (2015), West Virginia v. EPA dissent (2022).
Voting patterns: Reliably liberal on civil rights, criminal procedure, and
administrative law. Known for writing the most persuasive liberal dissents.
Notable trait: Former Solicitor General — approaches cases with practicality.
Known as a consensus-builder on the liberal wing.
""",
    'NMGorsuch': """
Justice Neil Gorsuch (appointed by Donald Trump, 2017).
Ideology: Conservative-libertarian. Martin-Quinn score: 1.99.
Religion: Episcopal.
Judicial philosophy: Gorsuch is a committed textualist and originalist in the
mold of Antonin Scalia. He is concerned with protecting individual liberty from
government overreach including administrative agencies. He authored Bostock v.
Clayton County holding Title VII protects LGBT employees based on pure textualism.
He is skeptical of Chevron deference and protective of Native American treaty rights.
Key opinions: Bostock v. Clayton County (2020), McGirt v. Oklahoma (2020).
Voting patterns: Reliably conservative but occasionally sides with liberals on
Fourth Amendment and criminal defendant cases. His textualism sometimes produces
liberal outcomes.
Notable trait: Most likely conservative to side with liberals on criminal
procedure. Independent streak makes him less predictable than Alito or Thomas.
""",
    'BMKavanaugh': """
Justice Brett Kavanaugh (appointed by Donald Trump, 2018).
Ideology: Center-right. Martin-Quinn score: 0.95.
Religion: Catholic.
Judicial philosophy: Kavanaugh is a mainstream conservative with institutionalist
tendencies similar to Roberts. He prioritizes precedent more than Thomas or Alito
and tends toward narrow rulings. He was a key vote refusing to overturn the ACA
in California v. Texas.
Key opinions: Biden v. Nebraska (2023), Allen v. Milligan (2023).
Voting patterns: Votes conservative on most issues but occasionally joins Roberts
and liberals to avoid extreme outcomes. More moderate on stare decisis.
Notable trait: Often provides the fifth vote alongside Roberts for narrow majority
opinions. Seen as pragmatic rather than ideological.
""",
    'ACBarrett': """
Justice Amy Coney Barrett (appointed by Donald Trump, 2020).
Ideology: Conservative. Martin-Quinn score: 1.73.
Religion: Catholic.
Judicial philosophy: Barrett is an originalist and textualist trained under
Antonin Scalia. She approaches cases methodically focusing on text and original
meaning. She has been more independent than expected, joining liberals in some
gun cases and writing separately from conservatives when she disagrees.
Key opinions: United States v. Rahimi (2024), Students for Fair Admissions (joined majority).
Voting patterns: Reliably conservative on abortion, administrative law, and
religious liberty. More moderate than Thomas and Alito on some procedural questions.
Notable trait: Academic background as Notre Dame professor makes her opinions
particularly methodical. Still establishing her jurisprudential identity.
""",
    'KBJackson': """
Justice Ketanji Brown Jackson (appointed by Joe Biden, 2022).
Ideology: Liberal. Martin-Quinn score: -3.11.
Religion: Protestant.
Judicial philosophy: Jackson is the Court's newest and most liberal justice.
As a former public defender and federal judge she brings a unique criminal
justice perspective. She is outspoken about race and the legacy of slavery,
notably in her Students for Fair Admissions dissent arguing the majority
ignored 14th Amendment history.
Key opinions: Students for Fair Admissions dissent (2023), Pulsifer v. United States (2024).
Voting patterns: Reliably votes with Sotomayor and Kagan. More willing to write
separately to advance bolder legal theories. Brings criminal defense perspective
that sometimes differs from other liberals on procedural cases.
Notable trait: First Black woman on the Supreme Court. Only current justice
with criminal defense experience. Known for aggressive oral argument questioning.
"""
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Define Oyez Fetch/match/clean functions -- Cell 2

from functools import lru_cache
import requests
import re
import time


@lru_cache(maxsize=None)
def fetch_term_cases(term):
    try:
        url = f"https://api.oyez.org/cases?per_page=300&filter=term:{int(term)}"
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return []

def fetch_oyez_summary(case_name, term):
    cases = fetch_term_cases(term)
    if not cases:
        return None
    clean_name = re.sub(r'[^\w\s]', '', case_name.lower()).strip()
    name_words = set(clean_name.split()) - {'v', 'vs', 'et', 'al', 'inc', 'llc', 'corp'}
    best_match = None
    best_score = 0
    for case in cases:
        oyez_name = case.get('name', '').lower()
        oyez_clean = re.sub(r'[^\w\s]', '', oyez_name).strip()
        oyez_words = set(oyez_clean.split()) - {'v', 'vs', 'et', 'al', 'inc', 'llc', 'corp'}
        score = len(name_words & oyez_words) / max(len(name_words), 1)
        if score > best_score:
            best_score = score
            best_match = case
    if best_match and best_score >= 0.4:
        desc = best_match.get('description', '') or ''
        question = best_match.get('question', '') or ''
        text = f"{desc} {question}".strip()
        text = re.sub(r'<[^>]+>', '', text).strip()
        if text:
            return text
    return None

In [15]:
# Get landmark case texts from Oyez -- Cell 3

landmark_cases = {
    'JGRoberts':  [('NFIB v. Sebelius', 2011),
                   ('Shelby County v. Holder', 2012),
                   ('Department of Homeland Security v. Regents', 2019)],
    'CThomas':    [("McDonald v. City of Chicago", 2009),
                   ("Dobbs v. Jackson Women's Health Organization", 2021),
                   ("New York State Rifle & Pistol Association v. Bruen", 2021)],
    'SAAlito':    [("Dobbs v. Jackson Women's Health Organization", 2021),
                   ("Burwell v. Hobby Lobby", 2013),
                   ("Janus v. AFSCME", 2017)],
    'SSotomayor': [("Obergefell v. Hodges", 2014),
                   ("Utah v. Strieff", 2015),
                   ("Trump v. Hawaii", 2017)],
    'EKagan':     [("Yates v. United States", 2014),
                   ("West Virginia v. EPA", 2021),
                   ("Dobbs v. Jackson Women's Health Organization", 2021)],
    'NMGorsuch':  [("Bostock v. Clayton County", 2019),
                   ("McGirt v. Oklahoma", 2019),
                   ("West Virginia v. EPA", 2021)],
    'BMKavanaugh':[("Biden v. Nebraska", 2022),
                   ("Allen v. Milligan", 2022),
                   ("Dobbs v. Jackson Women's Health Organization", 2021)],
    'ACBarrett':  [("United States v. Rahimi", 2023),
                   ("Students for Fair Admissions v. Harvard", 2022),
                   ("Dobbs v. Jackson Women's Health Organization", 2021)],
    'KBJackson':  [("Students for Fair Admissions v. Harvard", 2022),
                   ("Pulsifer v. United States", 2023),
                   ("United States v. Rahimi", 2023)]
}

justice_case_texts = {}
for code, cases in landmark_cases.items():
    texts = []
    for case_name, term in cases:
        text = fetch_oyez_summary(case_name, term)
        if text:
            texts.append(f"Case: {case_name}\n{text}")
        else:
            print(f"❌ {code}: {case_name} not found")
    justice_case_texts[code] = '\n\n'.join(texts)
    print(f"T {JUSTICE_NAMES[code]}: {len(texts)} cases fetched")

T Chief Justice John Roberts: 3 cases fetched
T Justice Clarence Thomas: 3 cases fetched
T Justice Samuel Alito: 3 cases fetched
T Justice Sonia Sotomayor: 3 cases fetched
T Justice Elena Kagan: 3 cases fetched
T Justice Neil Gorsuch: 3 cases fetched
T Justice Brett Kavanaugh: 3 cases fetched
T Justice Amy Coney Barrett: 3 cases fetched
T Justice Ketanji Brown Jackson: 3 cases fetched


In [16]:
# Bios and prominent opinions of each justice, sourced from supreme.justia.com -- Cell 4

justia_bios = {
    'JGRoberts': """Chief Justice John Roberts joined the U.S. Supreme Court on September 29, 2005, replacing Chief Justice William Rehnquist. Roberts was born in New York on January 27, 1955. He graduated summa cum laude from Harvard in 1976, majoring in history, and magna cum laude from Harvard Law School in 1979. He served as managing editor of the Harvard Law Review. Roberts clerked for Justice Rehnquist on the Supreme Court, served as Special Assistant to the Attorney General, Associate Counsel to President Reagan, and Principal Deputy Solicitor General from 1989 to 1993. The Senate confirmed him to the D.C. Circuit in 2003. Bush nominated Roberts to the Supreme Court in 2005 and the Senate confirmed him 78-22.

Although Roberts is relatively conservative, he sometimes has taken a more measured approach than the other conservative Justices. He wrote a concurring opinion in Dobbs that would have upheld the abortion restriction without overturning Roe v. Wade. During his confirmation hearings, Roberts explained that if it is not necessary to decide more to dispose of a case, in his view it is necessary not to decide more. In recent terms, he has struck down college affirmative action programs, radically reshaped the relationship between courts and government agencies, and voiced a broad view of presidential immunity.

SELECTED OPINIONS:
Trump v. United States (2024) - Separation of Powers: A former President is entitled to absolute immunity from criminal prosecution for actions within his conclusive and preclusive constitutional authority, and at least presumptive immunity for all official acts. There is no immunity for unofficial acts.
Loper Bright Enterprises v. Raimondo (2024) - Government Agencies: Chevron is overruled. Courts must exercise their independent judgment in deciding whether an agency has acted within its statutory authority and may not defer to an agency interpretation simply because a statute is ambiguous.
United States v. Rahimi (2024) - Gun Rights: An individual found by a court to pose a credible threat to the physical safety of another may be temporarily disarmed consistent with the Second Amendment.
Students for Fair Admissions v. Harvard (2023) - Equal Protection: College admissions programs violated the Equal Protection Clause when they lacked sufficiently focused and measurable objectives warranting the use of race, involved racial stereotyping, and lacked meaningful end points.
Biden v. Nebraska (2023) - Government Agencies: The authority to modify statutes and regulations allows an agency to make modest adjustments but not to transform them.
Moore v. Harper (2023) - Voting: The Elections Clause does not vest exclusive and independent authority in state legislatures to set the rules regarding federal elections.
Allen v. Milligan (2023) - Voting Rights: A district is not equally open as required by Section 2 of the Voting Rights Act when minority voters face bloc voting along racial lines that renders a minority vote unequal to a non-minority voter's vote.
West Virginia v. EPA (2022) - Environment: Congress did not grant the EPA the authority to devise emissions caps based on the generation-shifting approach taken in the Clean Power Plan.
DHS v. Regents of the University of California (2020) - Immigration: The DHS decision to rescind the DACA program was arbitrary and capricious under the Administrative Procedure Act.
Seila Law v. CFPB (2020) - Separation of Powers: An independent agency wielding significant executive power run by a single individual who cannot be removed by the President unless certain statutory criteria are met clashes with constitutional structure.
Trump v. Hawaii (2018) - Immigration: The President has broad discretion to suspend the entry of foreign nationals into the U.S.
Carpenter v. U.S. (2018) - Search and Seizure: The government's acquisition of an individual's cell-site records was a Fourth Amendment search.
Shelby County v. Holder (2013) - Voting Rights: Section 4 of the Voting Rights Act is unconstitutional and its formula can no longer be used as a basis for subjecting jurisdictions to preclearance.
NFIB v. Sebelius (2012) - Health Care: The individual health insurance mandate under the ACA was a permissible use of Congress' taxing power, but conditioning all Medicaid funding on states' compliance with a significant expansion was not valid.
Riley v. California (2014) - Search and Seizure: Without a warrant, police generally may not search digital information on a cell phone seized from an individual who has been arrested.""",

    'CThomas': """Justice Clarence Thomas joined the U.S. Supreme Court on October 23, 1991, replacing Justice Thurgood Marshall. He is the longest-serving current Justice by a wide margin. Thomas was born in Georgia on June 23, 1948. He graduated cum laude from Holy Cross in 1971 and received his law degree from Yale Law School in 1974. Over the next years, Thomas served as an Assistant Attorney General of Missouri, briefly worked for Monsanto, served on the Senate Commerce Committee, became Assistant Secretary for Civil Rights in the Department of Education, and spent eight years as Chairman of the Equal Employment Opportunity Commission. Bush nominated Thomas to the D.C. Circuit in 1989 and to the Supreme Court in 1991. The Senate confirmed him by a 52-48 vote after contentious hearings involving sexual harassment allegations by Anita Hill.

Thomas has earned a reputation as a deeply conservative Justice. He has generally opposed constitutional protections for abortion, LGBTQ+ rights, and affirmative action programs, while endorsing the right to bear arms under the Second Amendment. He often takes a narrow view of the statutory rights of employees and the constitutional rights of criminal defendants and prisoners. Thomas did not ask any questions during oral arguments between February 2006 and February 2016. His opinions show a greater willingness to revisit settled areas of law than many Justices — he frequently writes separately to argue for overturning precedents he considers wrongly decided.

SELECTED OPINIONS:
Garland v. Cargill (2024) - Gun Rights: A bump stock does not convert a semiautomatic rifle into a machinegun for the purposes of federal gun control laws.
New York State Rifle & Pistol Association v. Bruen (2022) - Gun Rights: When the Second Amendment's plain text covers an individual's conduct, the Constitution presumptively protects that conduct. To justify a firearm regulation, the government must demonstrate that the regulation is consistent with the nation's historical tradition of firearm regulation.
Little Sisters of the Poor v. Pennsylvania (2020) - Health Care: The Department of Health and Human Services may promulgate exemptions allowing organizations to exempt themselves from the ACA contraceptive coverage requirement on religious or moral grounds.
Kansas v. Glover (2020) - Search and Seizure: When an officer lacks information negating an inference that a vehicle is driven by its owner, a traffic stop after learning the registered owner's license has been revoked is reasonable.
Ohio v. American Express (2018) - Antitrust: Evidence of a price increase on one side of a two-sided transaction platform cannot by itself demonstrate an anti-competitive exercise of market power.
Star Athletica v. Varsity Brands (2017) - Copyrights: A feature incorporated into the design of a useful article is eligible for copyright protection only if it can be perceived as a two or three-dimensional work of art separate from the useful article.
Utah v. Strieff (2016) - Search and Seizure: The discovery of a valid pre-existing arrest warrant attenuated the connection between the unconstitutional investigatory stop and the evidence seized incident to a lawful arrest.
Reed v. Town of Gilbert (2015) - Free Speech: Content-based laws are presumptively unconstitutional and may be justified only if the government proves they are narrowly tailored to serve compelling state interests.
McDonald v. City of Chicago (2010) - Gun Rights: The Fourteenth Amendment incorporates the Second Amendment right to keep and bear arms for the purpose of self-defense against state interference.
Kansas v. Marsh (2006) - Death Penalty: A state death penalty statute may direct imposition of the death penalty when the state has proved beyond a reasonable doubt that mitigators do not outweigh aggravators.
Wilson v. Arkansas (1995) - Search and Seizure: The common-law knock and announce principle forms a part of the Fourth Amendment reasonableness inquiry.""",

    'SAAlito': """Justice Samuel Alito joined the U.S. Supreme Court on January 31, 2006, replacing Justice Sandra Day O'Connor. Alito was born in New Jersey on April 1, 1950. He graduated summa cum laude from Princeton University in 1972 and received his law degree from Yale Law School, where he was an editor of the Yale Law Journal. Alito clerked for the U.S. Court of Appeals for the Third Circuit, served as an Assistant U.S. Attorney in New Jersey, Assistant to the U.S. Solicitor General, Deputy Assistant Attorney General, and U.S. Attorney for the District of New Jersey. Bush nominated him to the Third Circuit in 1990 and to the Supreme Court in 2005. The Senate confirmed him by a 58-42 vote.

Alito holds a deeply conservative view of many issues. He has helped bolster gun rights and religious liberty while eroding voting rights and dissenting from decisions advancing LGBTQ+ rights. Alito's most significant legacy involves abortion — his majority opinion in Dobbs v. Jackson Women's Health Organization reversed Roe v. Wade. Alito wrote that the Constitution does not confer a right to abortion and the authority to regulate abortion must be returned to the people and their elected representatives.

SELECTED OPINIONS:
Dobbs v. Jackson Women's Health Organization (2022) - Abortion: The Constitution does not confer a right to abortion. Roe and Casey are overruled, and the authority to regulate abortion is returned to the people and their elected representatives.
Brnovich v. Democratic National Committee (2021) - Voting Rights: The mere fact that there is some disparity in impact does not necessarily mean that a system is not equally open or that it does not give everyone an equal opportunity to vote.
Janus v. AFSCME (2018) - Labor and Free Speech: The state's extraction of agency fees from non-consenting public-sector employees violates the First Amendment.
Burwell v. Hobby Lobby (2014) - Religion and Health Care: The Religious Freedom Restoration Act permits a closely held for-profit corporation to deny employees contraceptive coverage based on the religious objections of the corporation's owners.
Glossip v. Gross (2015) - Death Penalty: To succeed in an Eighth Amendment method of execution claim, a prisoner must establish that the method creates a demonstrated risk of severe pain substantially greater than known and available alternatives.
American Legion v. American Humanist Ass'n (2019) - Religion: When time's passage imbues a religiously expressive monument or symbol with familiarity and historical significance, removing it may no longer appear neutral. The passage of time gives rise to a strong presumption of constitutionality.
Sackett v. EPA (2023) - Environment: The Clean Water Act extends only to wetlands with a continuous surface connection to waters of the United States, so that they are indistinguishable from those waters.
Matal v. Tam (2017) - Free Speech and Trademarks: The disparagement provision of the Lanham Act prohibiting registration of trademarks that may disparage persons violates the Free Speech Clause.
McDonald v. City of Chicago (2010) - Gun Rights: Joined Thomas's opinion incorporating Second Amendment against states.
Vega v. Tekoh (2022) - Miranda Rights: A violation of Miranda rules does not provide a basis for a Section 1983 claim against a police officer who improperly obtained a statement.""",

    'SSotomayor': """Justice Sonia Sotomayor joined the U.S. Supreme Court on August 8, 2009, replacing Justice David Souter. She was the first Hispanic and the first woman of color to reach the Supreme Court. She was born in New York City on June 25, 1954, to parents who came from Puerto Rico. Sotomayor graduated summa cum laude from Princeton University in 1976 and received her law degree from Yale Law School in 1979, where she was an editor of the Yale Law Journal. She worked as an assistant district attorney in Manhattan, joined a private law firm, and served on several government boards. Bush nominated her to the U.S. District Court for the Southern District of New York in 1991. Clinton nominated her to the Second Circuit in 1997. She wrote nearly 400 majority opinions for the Second Circuit. Obama nominated her to the Supreme Court in 2009, and the Senate confirmed her 68-31.

Sotomayor generally takes a liberal position on controversial topics. Despite her service as a prosecutor, she has advocated vigorously for the rights of suspects and defendants. She has written some scathing dissents in an era when conservatives dominate the Court. She emphasizes lived experience and empathy in judicial decision-making.

SELECTED OPINIONS:
National Rifle Association of America v. Vullo (2024) - Free Speech: The First Amendment prohibits government officials from wielding their power selectively to punish or suppress speech, including through private intermediaries.
Andy Warhol Foundation v. Goldsmith (2023) - Copyrights: If an original work and secondary use share the same or highly similar purposes and the secondary use is commercial, the first fair use factor is likely to weigh against fair use.
Collins v. Virginia (2018) - Search and Seizure: The automobile exception does not permit the warrantless entry of a home or its curtilage to search a vehicle therein.
Perez v. Mortgage Bankers Ass'n (2015) - Government Agencies: Since an agency is not required to use notice-and-comment procedures to issue an initial interpretive rule, it is also not required to use those procedures to amend or repeal that rule.
Lane v. Franks (2014) - Labor and Free Speech: A public employee's sworn testimony outside the scope of their ordinary job duties is entitled to First Amendment protection.
J.D.B. v. North Carolina (2011) - Miranda Rights: The age of a child is relevant to the determination of whether they are in police custody for Miranda purposes.
Michigan v. Bryant (2011) - Criminal Trials: An identification and description of a shooter and the location of a shooting were not testimonial statements for Confrontation Clause purposes because they had a primary purpose to enable police assistance to meet an ongoing emergency.
Glossip v. Oklahoma (2025) - Criminal Trials: A criminal defendant is entitled to a new trial when the prosecution knowingly failed to correct false testimony and this error could have contributed to the verdict.
Catholic Charities Bureau v. Wisconsin (2025) - Religion: When the government distinguishes among religions based on theological differences in their provision of services, it imposes a denominational preference that must satisfy the highest level of judicial scrutiny.""",

    'EKagan': """Justice Elena Kagan joined the U.S. Supreme Court on August 7, 2010, replacing Justice John Paul Stevens. Kagan was born in New York City on April 28, 1960. She graduated summa cum laude from Princeton in 1981 and received a degree in politics from Oxford in 1983. She graduated magna cum laude from Harvard Law School in 1986, where she was a supervisory editor of the Harvard Law Review. Kagan clerked for Justice Thurgood Marshall, briefly practiced at a private firm, joined the University of Chicago faculty, worked for the Clinton administration as Associate White House Counsel and Deputy Assistant for Domestic Policy, became a professor and then Dean of Harvard Law School, and served as Solicitor General under Obama. Obama nominated her to the Supreme Court in 2010 and the Senate confirmed her 63-37. She was the first Justice since the 1970s who had never previously served as a judge.

Kagan is considered a liberal Justice on most key issues. She has voted in favor of the Affordable Care Act, same-sex marriage, gun control, and abortion, and she has sought to protect voting rights. Kagan has written some of the most readable opinions by the current Justices, emphasizing writing in words that non-lawyers can understand.

SELECTED OPINIONS:
Moody v. NetChoice (2024) - Free Speech: The First Amendment offers protection when an entity engaging in expressive activity is directed to accommodate messages it would prefer to exclude. A state may not interfere with private actors' speech to advance its own vision of ideological balance.
West Virginia v. EPA (2022 dissent) - Environment: Dissented from majority holding limiting EPA's authority to regulate carbon emissions under the Clean Air Act.
Counterman v. Colorado (2023) - Free Speech: Although true threats of violence are outside First Amendment protection, the First Amendment requires proof that the defendant had some subjective understanding of the threatening nature of their statements. The state only needs to prove recklessness.
Minerva Surgical v. Hologic (2021) - Patents: The doctrine of assignor estoppel applies only when the assignor's claim of invalidity contradicts explicit or implicit representations made in assigning the patent.
Ford Motor Co. v. Montana Eighth Judicial District Court (2021) - Lawsuits: When a company serves a market for a product in a state and that product causes an injury in the state, the state's courts may entertain the resulting lawsuit.
Kisor v. Wilkie (2019) - Government Agencies: A court should not afford deference to an agency's interpretation of its own regulations unless the regulation is genuinely ambiguous after exhausting all traditional tools of construction.
Miller v. Alabama (2012) - Death Penalty: The Eighth Amendment forbids a sentencing scheme that mandates life in prison without possibility of parole for juvenile homicide offenders.
Cooper v. Harris (2017) - Voting Rights: A state may not use race as the predominant factor in drawing district lines unless it has a compelling reason.""",

    'NMGorsuch': """Justice Neil Gorsuch joined the U.S. Supreme Court on April 10, 2017, replacing Justice Antonin Scalia. Gorsuch was born in Colorado on August 29, 1967. He graduated cum laude from Columbia University in 1988 and cum laude from Harvard Law School in 1991. He later received a doctorate in law from Oxford. Gorsuch clerked on the D.C. Circuit and for Justices Kennedy and White at the Supreme Court. He practiced law at a private firm until 2005, joined the Department of Justice as Principal Deputy Associate Attorney General, and was nominated by Bush to the Tenth Circuit in 2006, confirmed unanimously. Trump nominated Gorsuch to the Supreme Court on February 1, 2017. After Democrats sought to filibuster, Republicans changed Senate rules to allow confirmation by simple majority. Gorsuch was confirmed 54-45.

Gorsuch follows the textualist theory of legal interpretation, holding that laws should be interpreted according to the ordinary meaning of their words. Like his predecessor Scalia, he adheres to originalism — a constitutional provision should be given the meaning it would have had when it took effect. Although Gorsuch belongs to the conservative bloc, he repeatedly has joined the liberal Justices in support of Native American rights and occasionally on criminal procedure and Fourth Amendment cases.

SELECTED OPINIONS:
303 Creative LLC v. Elenis (2023) - Free Speech: The First Amendment prohibits a state from forcing a website designer to create expressive designs speaking messages with which the designer disagrees.
Kennedy v. Bremerton School District (2022) - Religion: The Free Exercise and Free Speech Clauses protect an individual engaging in a personal religious observance from government reprisal. The Constitution neither mandates nor permits the government to suppress such religious expression.
Bostock v. Clayton County (2020) - Labor and LGBTQ+ Rights: An employer that fires an individual merely for being gay or transgender violates Title VII.
McGirt v. Oklahoma (2020) - Native American Rights: Land promised to a tribe by the U.S. government remained an Indian reservation for the purposes of federal criminal law.
Ramos v. Louisiana (2020) - Criminal Trials: The Sixth Amendment right to a jury trial requires a unanimous verdict to convict a defendant of a serious offense.
NCAA v. Alston (2021) - Antitrust: The NCAA is not immune from the Sherman Act because its restrictions happen to fall at the intersection of higher education, sports, and money.
Bucklew v. Precythe (2019) - Death Penalty: To establish that a state's chosen execution method cruelly superadds pain, a prisoner must show a feasible and readily implemented alternative method that would significantly reduce a substantial risk of severe pain.
Bondi v. Vanderstok (2025) - Gun Rights: The Gun Control Act of 1968 permits the ATF to regulate some weapon parts kits and unfinished frames or receivers.""",

    'BMKavanaugh': """Justice Brett Kavanaugh joined the U.S. Supreme Court on October 6, 2018, replacing Justice Anthony Kennedy. Kavanaugh was born in Washington, D.C. on February 12, 1965. He graduated cum laude from Yale University in 1987 and from Yale Law School in 1990. He clerked on the Third Circuit, Ninth Circuit, and for Justice Kennedy at the Supreme Court. He served as Associate Counsel in the Office of Independent Counsel Kenneth Starr and contributed to the 1998 Starr Report urging Clinton's impeachment. After joining the legal effort in the 2000 Florida recount, he served in the Bush White House Counsel's Office and as Staff Secretary. Bush nominated Kavanaugh to the D.C. Circuit in 2003, confirmed in 2006. Trump nominated him to the Supreme Court on July 10, 2018. His confirmation hearings were contentious, involving sexual misconduct allegations. The Senate confirmed him 50-48.

Kavanaugh has been described as the median Justice — there are four Justices more liberal and four more conservative. He is clearly a conservative jurist but takes a more moderate approach than Thomas, Alito, or Gorsuch on some issues. In June 2022 he voted with four other conservatives to strike down Roe v. Wade, declining to join Roberts in a more moderate position.

SELECTED OPINIONS:
Food and Drug Administration v. Alliance for Hippocratic Medicine (2024) - Abortion: Sincere legal, moral, ideological, and policy objections to elective abortion and to the FDA's relaxed regulation of an abortion drug alone do not establish a justiciable case or controversy in federal court.
Moore v. U.S. (2024) - Taxes: Congress may attribute an entity's realized and undistributed income to the entity's shareholders and then tax the shareholders on their portions of that income.
American Hospital Ass'n v. Becerra (2022) - Government Agencies: There is a strong presumption in favor of judicial review of final agency action.
TransUnion LLC v. Ramirez (2021) - Role of Courts: Only plaintiffs concretely harmed by a defendant's statutory violation have Article III standing to seek damages against that private defendant in federal court.
Jones v. Mississippi (2021) - Death Penalty: A sentencer need not make a separate factual finding of permanent incorrigibility before sentencing a murderer under 18 to life without parole.
FCC v. Prometheus Radio Project (2021) - Government Agencies: The APA's arbitrary-and-capricious standard requires that agency action be reasonable and reasonably explained.
Manhattan Community Access Corp. v. Halleck (2019) - Free Speech: When a private entity operates public access channels on a cable system, it is not performing a traditional public function and is not subject to First Amendment constraints on its editorial discretion.
Becerra v. Braidwood Management (2025) - Separation of Powers: Members of the U.S. Preventive Services Task Force are inferior executive officers who may be appointed by the Secretary of Health and Human Services.""",

    'ACBarrett': """Justice Amy Coney Barrett joined the U.S. Supreme Court on October 27, 2020, replacing Justice Ruth Bader Ginsburg. Barrett was born in Louisiana on January 28, 1972. She graduated magna cum laude from Rhodes College in 1994 and summa cum laude and as valedictorian from Notre Dame Law School in 1997, where she was executive editor of the Notre Dame Law Review. Barrett clerked on the D.C. Circuit and for Justice Antonin Scalia. She briefly worked at a private law firm before joining Notre Dame Law School as a professor in 2002. Trump nominated her to the Seventh Circuit in 2017, confirmed in October. Trump nominated her to the Supreme Court on September 29, 2020. Republicans expedited her confirmation before the 2020 election. The Senate confirmed her 52-48, the first Justice since the 19th century to receive confirmation without any votes from the minority party.

Barrett resembles Gorsuch in her adherence to statutory textualism and constitutional originalism. She believes these schools of thought, both associated with Scalia, are closely related. She has explained that textualism insists that judges must construe statutory language consistent with its ordinary meaning, and that originalism holds that constitutional text means what it did at the time it was ratified and that this original public meaning is authoritative.

SELECTED OPINIONS:
Trump v. CASA Inc. (2025) - Role of Courts: Universal injunctions in which district courts prohibit enforcement of a law or policy against anyone likely exceed the equitable authority that Congress has granted to federal courts.
Corner Post v. Board of Governors (2024) - Government Agencies: An APA claim does not accrue for the purposes of the six-year statute of limitations until the plaintiff is injured by final agency action.
Sheetz v. County of El Dorado (2024) - Property Rights: The Takings Clause of the Fifth Amendment prohibits legislatures and agencies alike from imposing unconstitutional conditions on land-use permits.
Lindke v. Freed (2024) - Free Speech: When a government official posts about job-related topics on social media, this speech is attributable to the government only if the official possessed actual authority to speak on the government's behalf and purported to exercise that authority.
Haaland v. Brackeen (2023) - Powers of Congress: When Congress validly legislates pursuant to its Article I powers, conflicting state family law is preempted. Congress may impose ancillary recordkeeping requirements related to state-court proceedings without violating the Tenth Amendment.
United States v. Rahimi (2024) - Gun Rights: An individual found by a court to pose a credible threat to the physical safety of another may be temporarily disarmed consistent with the Second Amendment.
Department of State v. Munoz (2024) - Immigration: A citizen does not have a fundamental liberty interest in their noncitizen spouse being admitted to the country.""",

    'KBJackson': """Justice Ketanji Brown Jackson joined the U.S. Supreme Court on June 30, 2022, replacing Justice Stephen Breyer. Jackson was born in Washington, D.C. on September 14, 1970. She graduated magna cum laude from Harvard University in 1992 and cum laude from Harvard Law School in 1996, where she was a supervising editor of the Harvard Law Review. Jackson clerked on the U.S. District Court for the District of Massachusetts, the U.S. Court of Appeals for the First Circuit, and for Justice Stephen Breyer at the Supreme Court. She served as an assistant special counsel to the U.S. Sentencing Commission and as an assistant federal public defender. Obama nominated her to the U.S. District Court for the District of Columbia in 2012, confirmed in 2013. Biden nominated her to the D.C. Circuit in 2021, confirmed in June. Biden nominated her to the Supreme Court on February 28, 2022. After the Senate Judiciary Committee deadlocked 11-11, the Senate confirmed her 53-47. She is the first woman of color to sit on the Supreme Court and the only current Justice with criminal defense experience.

Jackson emphasizes a three-step methodology to preserve impartiality: proceeding from a position of neutrality, evaluating the facts and appropriate materials, and interpreting and applying the law to the facts. She has been outspoken about race and the legacy of slavery in American law, notably in her Students for Fair Admissions dissent. She brings a criminal defense perspective that sometimes differs from other liberal Justices on procedural criminal cases.

SELECTED OPINIONS:
Ames v. Ohio Department of Youth Services (2025) - Labor: For a prima facie case of disparate treatment employment discrimination under Title VII, a plaintiff who is a member of a majority group does not need to show background circumstances to support the suspicion that the defendant is an unusual employer who discriminates against the majority.
Students for Fair Admissions v. Harvard (2023 dissent) - Equal Protection: Dissented from the majority's elimination of race-conscious admissions, arguing the majority ignored the constitutional history of the 14th Amendment and its purpose of remedying racial inequality.
Pulsifer v. United States (2024) - Criminal Sentencing: Wrote opinion interpreting the First Step Act's safety valve provision for drug offenders."""
}

print(f"Total justices: {len(justia_bios)}")
for code, bio in justia_bios.items():
    print(f"  {JUSTICE_NAMES[code]}: {len(bio.split())} words")

Total justices: 9
  Chief Justice John Roberts: 708 words
  Justice Clarence Thomas: 614 words
  Justice Samuel Alito: 529 words
  Justice Sonia Sotomayor: 512 words
  Justice Elena Kagan: 480 words
  Justice Neil Gorsuch: 456 words
  Justice Brett Kavanaugh: 461 words
  Justice Amy Coney Barrett: 456 words
  Justice Ketanji Brown Jackson: 365 words


In [17]:
# Import sentence transformer -- Cell 5

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer('all-mpnet-base-v2')
print("Embed model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embed model loaded


In [18]:
# Combine justia bios + Manual profiles + Oyez case texts -- Cell 6

enhanced_profiles = {}
for code in CURRENT_JUSTICES:
    justia = justia_bios.get(code, '')
    manual = justice_profiles.get(code, '').strip()
    cases  = justice_case_texts.get(code, '')

    enhanced_profiles[code] = f"""
{justia}

JUDICIAL PHILOSOPHY AND VOTING PATTERNS:
{manual}

ADDITIONAL LANDMARK CASES FROM OYEZ:
{cases}
""".strip()

    print(f"{JUSTICE_NAMES[code]}: {len(enhanced_profiles[code].split())} words total")

# Re-embed and save
rag_chunks = []
for code in CURRENT_JUSTICES:
    rag_chunks.append({
        'justice_code': code,
        'justice_name': JUSTICE_NAMES[code],
        'chunk_idx': 0,
        'text': enhanced_profiles[code]
    })

chunk_texts = [c['text'] for c in rag_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=32,
                                       show_progress_bar=True,
                                       convert_to_numpy=True)
print(f"Embeddings: {chunk_embeddings.shape}")

with open(f'{BASE}/rag_chunks.pkl', 'wb') as f:
    pickle.dump(rag_chunks, f)
np.save(f'{BASE}/chunk_embeddings.npy', chunk_embeddings)
with open(f'{BASE}/combined_profiles.pkl', 'wb') as f:
    pickle.dump(enhanced_profiles, f)

print("Saved")

Chief Justice John Roberts: 1180 words total
Justice Clarence Thomas: 940 words total
Justice Samuel Alito: 844 words total
Justice Sonia Sotomayor: 884 words total
Justice Elena Kagan: 777 words total
Justice Neil Gorsuch: 769 words total
Justice Brett Kavanaugh: 744 words total
Justice Amy Coney Barrett: 741 words total
Justice Ketanji Brown Jackson: 697 words total


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings: (9, 768)
Saved


In [19]:
# Download -- Cell 9

from google.colab import files

files.download(f'{BASE}/combined_profiles.pkl')
files.download(f'{BASE}/rag_chunks.pkl')
files.download(f'{BASE}/chunk_embeddings.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>